In [ ]:
from text_pipeline import load_and_prepare_sequence_data
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score
import os
import gc

FEATURE_CACHE_DIR = "/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/cache/tinyllama_features"

ACCUM_STEPS = 8

BOTTLENECK = {
    "base": None,
    "small": None,
    "tiny": 128
}

def run_tinyllama(freeze_encoder: bool, mode: int, size="base", window=None):
    MODEL_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
    device = torch.device("cuda")

    suffix = f"mode{mode}_w{window}_len256_{MODEL_NAME.split('/')[-1]}"

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available")

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    train_dataset, val_dataset, test_dataset, label2id, id2label, class_weights = load_and_prepare_sequence_data(
        mode,
        MODEL_NAME,
        max_length=256,
        window=window
    )


    def masked_mean_pooling(hidden_states, attention_mask):
        mask = attention_mask.unsqueeze(-1).to(hidden_states.dtype)
        masked = hidden_states * mask
        summed = masked.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-6)
        return summed / counts

    num_labels = len(label2id)

    if freeze_encoder:
      print("Running frozen")

      os.makedirs(FEATURE_CACHE_DIR, exist_ok=True)

      train_path = os.path.join(FEATURE_CACHE_DIR, f"train_features_{suffix}.pt")
      val_path = os.path.join(FEATURE_CACHE_DIR, f"val_features_{suffix}.pt")
      test_path = os.path.join(FEATURE_CACHE_DIR, f"test_features_{suffix}.pt")

      def make_loader(ds):
        return DataLoader(
            ds,
            batch_size=64,
            shuffle=False,
            num_workers=0,
            pin_memory=True
        )

      @torch.no_grad()
      def precompute_split(loader, split_name, encoder):
          all_feats = []
          all_labels = []

          for batch in tqdm(loader, desc=f"Precompute {split_name}"):
              input_ids = batch["input_ids"].to(device, non_blocking=True)
              attention_mask = batch["attention_mask"].to(device, non_blocking=True)
              labels = batch["labels"]

              outputs = encoder(input_ids=input_ids, attention_mask=attention_mask)
              pooled = masked_mean_pooling(outputs.last_hidden_state, attention_mask)
              pooled = pooled.float().cpu()

              all_feats.append(pooled)
              all_labels.append(labels.cpu())

          if len(all_feats) == 0:
              raise RuntimeError(f"No features computed for {split_name} split")

          if len(all_labels) == 0:
              raise RuntimeError(f"No labels computed for {split_name} split")

          X = torch.cat(all_feats, dim=0)
          y = torch.cat(all_labels, dim=0)
          return X, y

      encoder = None

      if os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(test_path):
        print("Loading precomputed features")
        train_X, train_y = torch.load(train_path, map_location="cpu")
        val_X, val_y = torch.load(val_path, map_location="cpu")
        test_X, test_y = torch.load(test_path, map_location="cpu")

        train_X = train_X.float()
        val_X = val_X.float()
        test_X = test_X.float()

        gc.collect()

      else:
        print("Precomputing and saving features")
        encoder = AutoModel.from_pretrained(MODEL_NAME).to(device)
        encoder.eval()

        for p in encoder.parameters():
            p.requires_grad = False

        train_loader = make_loader(train_dataset)
        val_loader = make_loader(val_dataset)
        test_loader = make_loader(test_dataset)

        train_X, train_y = precompute_split(train_loader, "train", encoder)
        val_X, val_y = precompute_split(val_loader, "val", encoder)
        test_X, test_y = precompute_split(test_loader, "test", encoder)

        torch.save((train_X, train_y), train_path)
        torch.save((val_X, val_y), val_path)
        torch.save((test_X, test_y), test_path)

        del encoder
        del train_loader, val_loader, test_loader
        gc.collect()
        torch.cuda.empty_cache()

      hidden_size = train_X.shape[1]
      bottleneck_dim = BOTTLENECK[size]

      class FeatureClassifier(nn.Module):
        def __init__(self):
          super().__init__()
          self.norm = nn.LayerNorm(hidden_size)

          if size == "tiny":
            self.classifier = nn.Sequential(
                nn.Linear(hidden_size, bottleneck_dim),
                nn.ReLU(),
                nn.Linear(bottleneck_dim, num_labels)
            )
          else:
            self.classifier = nn.Linear(hidden_size, num_labels)

        def forward(self, x):
          x = self.norm(x)
          return self.classifier(x)

      model = FeatureClassifier().to(device)

      criterion = nn.CrossEntropyLoss(weight=class_weights.to(device).float())
      optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

      def make_feat_loader(X, y, shuffle):
        return DataLoader(
            TensorDataset(X, y),
            batch_size=128,
            shuffle=shuffle,
            num_workers=0,
            pin_memory=True
        )

      train_loader = make_feat_loader(train_X, train_y, shuffle=True)
      val_loader = make_feat_loader(val_X, val_y, shuffle=False)
      test_loader = make_feat_loader(test_X, test_y, shuffle=False)

      def run_epoch(loader, train=True):
          if train:
              model.train()
          else:
              model.eval()

          total_loss = 0.0
          total_correct = 0
          total_samples = 0
          all_logits = []
          all_labels = []

          context = torch.enable_grad() if train else torch.no_grad()
          with context:
              if train:
                  optimizer.zero_grad(set_to_none=True)
              for feats, labels in tqdm(loader, desc="Train" if train else "Eval"):
                  feats = feats.to(device, non_blocking=True)
                  labels = labels.to(device, non_blocking=True)

                  logits = model(feats)
                  loss = criterion(logits, labels)

                  if train:
                      loss.backward()
                      torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                      optimizer.step()
                      optimizer.zero_grad(set_to_none=True)

                  preds = torch.argmax(logits, dim=1)
                  total_loss += loss.item()
                  total_correct += (preds == labels).sum().item()
                  total_samples += labels.size(0)

                  all_logits.append(logits.detach().cpu())
                  all_labels.append(labels.detach().cpu())

          all_logits = torch.cat(all_logits)
          all_labels = torch.cat(all_labels)
          preds = torch.argmax(all_logits, dim=1)

          return {
              "loss": total_loss / len(loader),
              "accuracy": total_correct / total_samples,
              "logits": all_logits,
              "labels": all_labels,
              "macro_f1": f1_score(all_labels.numpy(), preds.numpy(), average="macro")
          }

      train_losses, val_losses = [], []

      for epoch in range(1):
        train_epoch = run_epoch(train_loader, train=True)
        val_epoch = run_epoch(val_loader, train=False)

        train_losses.append(train_epoch["loss"])
        val_losses.append(val_epoch["loss"])

        print(
            f"Epoch {epoch+1} | "
            f"train_loss={train_epoch['loss']:.4f} | "
            f"val_loss={val_epoch['loss']:.4f} | "
            f"val_acc={val_epoch['accuracy']:.4f} | "
            f"val_macro_f1={val_epoch['macro_f1']:.4f}"
        )

      test_epoch = run_epoch(test_loader, train=False)

    else:
      print("Running unfrozen, full fine tuning")

      train_dataset = torch.utils.data.Subset(train_dataset, range(min(300000, len(train_dataset))))
      val_dataset = torch.utils.data.Subset(val_dataset, range(min(30000, len(val_dataset))))
      test_dataset = torch.utils.data.Subset(test_dataset, range(min(30000, len(test_dataset))))

      tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

      if tokenizer.pad_token is None:
          tokenizer.pad_token = tokenizer.eos_token

      base_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
      base_model.gradient_checkpointing_enable()
      base_model.config.use_cache = False

      class TinyLlamaClassifier(nn.Module):
        def __init__(self):
          super().__init__()
          hidden_size = base_model.config.hidden_size
          bottleneck_dim = BOTTLENECK[size]

          self.encoder = base_model
          self.norm = nn.LayerNorm(hidden_size)

          if size == "tiny":
            self.classifier = nn.Sequential(
                nn.Linear(hidden_size, bottleneck_dim),
                nn.ReLU(),
                nn.Linear(bottleneck_dim, num_labels)
            )
          else:
            self.classifier = nn.Linear(hidden_size, num_labels)

        def forward(self, input_ids, attention_mask):
          outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
          pooled = masked_mean_pooling(outputs.last_hidden_state, attention_mask)
          pooled = self.norm(pooled)
          return self.classifier(pooled)

      model = TinyLlamaClassifier().to(device)

      criterion = nn.CrossEntropyLoss(weight=class_weights.to(device).float())

      optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)

      train_loader = DataLoader(
          train_dataset,
          batch_size=8,
          shuffle=True,
          num_workers=0,
          pin_memory=True
      )

      val_loader = DataLoader(
          val_dataset,
          batch_size=8,
          shuffle=False,
          num_workers=0,
          pin_memory=True
      )

      test_loader = DataLoader(
          test_dataset,
          batch_size=8,
          shuffle=False,
          num_workers=0,
          pin_memory=True
      )

      def run_epoch(loader, train=True):
        model.train() if train else model.eval()

        total_loss, total_correct, total_samples = 0.0, 0, 0
        all_logits, all_labels = [], []

        context = torch.enable_grad() if train else torch.no_grad()
        with context:
          if train:
            optimizer.zero_grad(set_to_none=True)
          last_step = -1
          for step, batch in enumerate(tqdm(loader, desc="Train" if train else "Eval")):
            last_step = step
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            if train:
              loss = loss / ACCUM_STEPS
              loss.backward()
              torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
              if (step + 1) % ACCUM_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            preds = torch.argmax(logits, dim=1)
            total_loss += loss.item()
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
            all_logits.append(logits.detach().cpu())
            all_labels.append(labels.detach().cpu())

        if train and last_step >= 0 and (last_step + 1) % ACCUM_STEPS != 0:
          optimizer.step()
          optimizer.zero_grad(set_to_none=True)

        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels)

        return {
          "loss": total_loss / len(loader),
          "accuracy": total_correct / total_samples,
          "logits": all_logits,
          "labels": all_labels,
          "macro_f1": f1_score(all_labels.numpy(), torch.argmax(all_logits, dim=1).numpy(), average="macro")
        }

      train_losses, val_losses = [], []

      for epoch in range(2):
        train_epoch = run_epoch(train_loader, train=True)
        val_epoch = run_epoch(val_loader, train=False)

        train_losses.append(train_epoch["loss"])
        val_losses.append(val_epoch["loss"])

        print(
            f"Epoch {epoch+1} | "
            f"train_loss={train_epoch['loss']:.4f} | "
            f"val_loss={val_epoch['loss']:.4f} | "
            f"val_acc={val_epoch['accuracy']:.4f} | "
            f"val_macro_f1={val_epoch['macro_f1']:.4f}"
        )

      test_epoch = run_epoch(test_loader, train=False)

    preds = torch.argmax(test_epoch["logits"], dim=1).numpy()
    labels = test_epoch["labels"].numpy()

    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    cm = confusion_matrix(labels, preds)

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "train_loss": train_losses,
        "val_loss": val_losses,
        "confusion_matrix": cm,
        "history": val_losses,
        "test_logits": test_epoch["logits"].numpy(),
        "test_labels": test_epoch["labels"].numpy()
    }

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ModuleNotFoundError: No module named 'text_pipeline'